In [ ]:
#calculate the solar and eolic potential for each parroqia and canton and save to csv
#suitability raster solar: C:\HyGalapagos\outputs_solar\cost_raster_24_04.tif
#suitibility raster eolic: C:\HyGalapagos\outputs_eolico\cost_raster_24_04.tif

# isabela: "C:\Galapagos_IERSE\01_Organizacion_Territorial\solo_isabela.shp"
# floreana : "C:\Galapagos_IERSE\01_Organizacion_Territorial\floreana.shp"
# santacruz : "C:\Galapagos_IERSE\01_Organizacion_Territorial\santa_cruz_baltra.shp"
# sancristobal : "C:\Galapagos_IERSE\01_Organizacion_Territorial\san_cristobal.shp"


#pontential conversion solar = solar 1.3 ha por 1MW solar
#potential conversion eolic = 0.9ha per MW

In [5]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask

ROOT = Path(r"C:/HyGalapagos")
BASE_ROOT = Path(r"C:/Galapagos_IERSE/01_Organizacion_Territorial")
SOLAR_RASTER = ROOT / "outputs_solar/cost_raster_24_04.tif"
EOLIC_RASTER = ROOT / "outputs_eolico/cost_raster_24_04.tif"
FULL_PARK = BASE_ROOT / "Parroquias_Galapagos_CONALI_2018.shp"
ISLAND_FILES = {
    "Isabela": BASE_ROOT / "solo_isabela.shp",
    "Floreana": BASE_ROOT / "floreana.shp",
    "Santa Cruz": BASE_ROOT / "santa_cruz_baltra.shp",
    "San Cristobal": BASE_ROOT / "san_cristobal.shp",
}
OUT_DIR = ROOT / "outputs_solar/potential_reports_islands"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOLAR_HA_PER_MW = 1.3
EOLIC_HA_PER_MW = 0.9


def zonal_area_and_mean(geodf, raster_path):
    with rasterio.open(raster_path) as src:
        if geodf.crs != src.crs:
            geodf = geodf.to_crs(src.crs)

        pixel_area_ha = abs(src.transform.a * src.transform.e) / 10000.0
        areas_ha = []
        means = []

        for geom in geodf.geometry:
            try:
                data, _ = mask(src, [geom], crop=True, filled=False)
                arr = data[0]
                valid = np.isfinite(arr) & (~arr.mask)
                positive = valid & (arr > 0)

                count = int(np.sum(positive))
                area_ha = count * pixel_area_ha
                mean_val = float(np.mean(arr[positive])) if count > 0 else np.nan
            except ValueError:
                area_ha = 0.0
                mean_val = np.nan

            areas_ha.append(area_ha)
            means.append(mean_val)

    out = geodf.copy()
    out["suitable_area_ha"] = areas_ha
    out["suitability_mean"] = means
    return out


def add_potential_fields(gdf, ha_per_mw, tech):
    out = gdf.copy()
    out[f"potential_mw_{tech}"] = out["suitable_area_ha"] / ha_per_mw
    return out


def build_islands_gdf():
    parts = []
    for island_name, shp in ISLAND_FILES.items():
        island = gpd.read_file(shp)
        island = island[["geometry"]].copy()
        island["island"] = island_name
        island = island.dissolve(by="island", as_index=False)
        parts.append(island)
    return gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), geometry="geometry", crs=parts[0].crs)


def plot_potential_map(base_gdf, overlay_gdf, value_col, title, path_png):
    if base_gdf.crs != overlay_gdf.crs:
        base_gdf = base_gdf.to_crs(overlay_gdf.crs)

    fig, ax = plt.subplots(figsize=(11, 9))

    base_gdf.plot(ax=ax, color="#e8efe9", edgecolor="#9aa4a6", linewidth=0.8)

    overlay_gdf.plot(
        column=value_col,
        cmap="YlOrRd",
        linewidth=0.0,
        legend=True,
        legend_kwds={"label": f"{value_col} (MW)", "shrink": 0.75},
        ax=ax,
    )

    colors = {
        "Isabela": "#1f77b4",
        "Floreana": "#2ca02c",
        "Santa Cruz": "#d62728",
        "San Cristobal": "#9467bd",
    }
    outline_handles = []
    potential_handles = []
    for island_name, color in colors.items():
        part = overlay_gdf[overlay_gdf["island"] == island_name]
        if len(part) > 0:
            part.boundary.plot(ax=ax, color=color, linewidth=2.0)
            outline_handles.append(Line2D([0], [0], color=color, lw=2.5, label=island_name))

            rounded_mw = int(round(float(part.iloc[0][value_col])))
            point = part.geometry.iloc[0].representative_point()
            ax.text(
                point.x,
                point.y,
                f"{island_name}\n{rounded_mw} MW",
                fontsize=9,
                ha="center",
                va="center",
                color="#1f1f1f",
                bbox={"facecolor": "white", "edgecolor": color, "alpha": 0.8, "boxstyle": "round,pad=0.25"},
            )

            potential_handles.append(Patch(facecolor="none", edgecolor="none", label=f"{island_name}: {rounded_mw} MW"))

    legend1 = ax.legend(handles=outline_handles, title="Island outlines", loc="lower left", frameon=True)
    ax.add_artist(legend1)
    ax.legend(handles=potential_handles, title="Rounded potential", loc="lower right", frameon=True)
    ax.set_title(title)
    ax.set_axis_off()
    plt.tight_layout()
    fig.savefig(path_png, dpi=220)
    plt.close(fig)


full_park = gpd.read_file(FULL_PARK)
islands = build_islands_gdf()

solar_islands = zonal_area_and_mean(islands, SOLAR_RASTER)
solar_islands = add_potential_fields(solar_islands, SOLAR_HA_PER_MW, "solar")

eolic_islands = zonal_area_and_mean(islands, EOLIC_RASTER)
eolic_islands = add_potential_fields(eolic_islands, EOLIC_HA_PER_MW, "eolic")

solar_islands[["island", "suitable_area_ha", "suitability_mean", "potential_mw_solar"]].to_csv(
    OUT_DIR / "solar_islands_potential.csv", index=False
)

eolic_islands[["island", "suitable_area_ha", "suitability_mean", "potential_mw_eolic"]].to_csv(
    OUT_DIR / "eolic_islands_potential.csv", index=False
)

plot_potential_map(
    full_park,
    solar_islands,
    "potential_mw_solar",
    "Solar potential map - selected Galapagos islands",
    OUT_DIR / "map_solar_islands.png",
)

plot_potential_map(
    full_park,
    eolic_islands,
    "potential_mw_eolic",
    "Eolic potential map - selected Galapagos islands",
    OUT_DIR / "map_eolic_islands.png",
)

print("Done. Island outputs exported to:", OUT_DIR)

Done. Island outputs exported to: C:\HyGalapagos\outputs_solar\potential_reports_islands
